In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(r"C:\Users\Soli1\SIGMOD\Solmaz\LSH-APG\cppCode\LSH-APG")
IDX = ROOT / "indexes"
OUT = ROOT / "github_tables"
OUT.mkdir(exist_ok=True)

files = {
    "Full DAPG, local + global": IDX / "solmaz_table_o2_full_local_global_sift_W600p000.csv",
    "Global-only approx, pC=0.999": IDX / "solmaz_table_o2_global_only_p999_sift_W600p000.csv",
    "Local-only relaxed cap, T=200": IDX / "solmaz_table_o2_local_only_relaxedcap_sift_W600p000.csv",
}

rows = []

for variant, path in files.items():
    df = pd.read_csv(path)

    
    if "MaxRecall" in df.columns:
        mask = df["MaxRecall"].isna() & df["SearchRt0p99_ms"].notna()
        df.loc[mask, "MaxRecall"] = df.loc[mask, "SearchRt0p99_ms"]
        df.loc[mask, "MaxRecallEf"] = df.loc[mask, "SearchRt0p995_ms"]
        df.loc[mask, "SearchRt0p99_ms"] = np.nan
        df.loc[mask, "SearchRt0p995_ms"] = np.nan

    dapg = df[df["SolmazMethod"] == "DAPG"].iloc[0]

    rows.append({
        "Variant": variant,
        "Local percentile pruning": "Yes" if "Global-only" not in variant else "Approx. No",
        "Global cap": "Relaxed" if "Local-only" in variant else "Yes",
        "T": dapg["T"],
        "MaxRecall": dapg["MaxRecall"],
        "Ef(MaxRecall)": dapg["MaxRecallEf"],
        "SearchRt@0.95(ms)": dapg["SearchRt0p95_ms"],
        "SearchRt@0.97(ms)": dapg["SearchRt0p97_ms"],
        "IndexingTime(s)": dapg["IndexingTime_s"],
        "InsertAvg(ms)": dapg["InsertAvg_ms"],
        "DeleteAvg(ms)": dapg["DeleteAvg_ms"],
    })

ablation = pd.DataFrame(rows)

display(ablation)

,Variant,Local percentile pruning,Global cap,T,MaxRecall,Ef(MaxRecall),SearchRt@0.95(ms),SearchRt@0.97(ms),IndexingTime(s),InsertAvg(ms),DeleteAvg(ms)
0,"Full DAPG, local + global",Yes,Yes,24,0.9772,1960.0,0.88466,1.95520,78.1467,0.644445,0.149730
1,"Global-only approx, pC=0.999",Approx. No,Yes,24,0.9768,2080.0,0.88044,1.81279,77.9545,0.654255,0.094695
2,"Local-only relaxed cap, T=200",Yes,Relaxed,200,0.9769,2060.0,1.73639,3.59185,199.4710,1.974030,1.241030
